## Generating summary & bias for each article:

In [3]:
import sys
import pandas as pd
import csv
import os
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
from smc_steer import bias_model_factory, TwistModel, gen_summary

In [4]:
sys.path.append("/data/cb/scratch/bfefferm/NLP-Project/hfppl")

**Loading Dataset:**

In [5]:
dataset = pd.read_csv('../POLITICS_finetuning/processed_data.csv')

In [6]:
dataset

,title,body,stance
0,"Ryan goes on offense over Medicare, accuses Ob...",Paul Ryan went on offense Tuesday in response ...,center
1,Obama Medicare Attack In 2008 Targeted McCain ...,WASHINGTON -- In the wake of Mitt Romney's cho...,right
2,Clintons Report Earnings of $139 Million in Se...,Hillary Rodham Clinton on Friday released her ...,left
3,Clinton camp releases candidate's clean bill o...,WASHINGTON – Hillary Clinton's presidential ca...,center
4,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
...,...,...,...
295,Chris Christie Announces He's Running For Pres...,Chris Christie Announces He's Running For Pres...,right
296,"Bloomberg says Trump a 'dangerous demagogue,' ...",Former New York City Mayor Michael Bloomberg f...,conservative
297,Donald Trump Says Joe Lieberman Is His Top Cho...,WASHINGTON ― President Donald Trump is “very c...,liberal
298,Trump's 'Compromise' Immigration Offer To Demo...,WASHINGTON ― President Donald Trump’s offer to...,liberal


**Specifying file paths:**

In [7]:
# Model paths:
bias_model_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

# Specifying model name:
llm_model_name = 'gpt2'

# Loading llm:
llm = CachedCausalLM.from_pretrained(llm_model_name)

2023-11-18 21:23:48.078849: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-11-18 21:23:48.078896: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-11-18 21:23:48.078929: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-11-18 21:23:48.087556: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-18 21:23:49.919978: W tensorflow/c

You are loading your model in 8bit or 4bit but no linear modules were found in your model. this can happen for some architectures such as gpt2 that uses Conv1D instead of Linear layers. Please double check your model architecture, or submit an issue on github if you think this is a bug.


**Iterating over each summary, predicting bias, & saving to `.csv`:**

In [ ]:
num_summaries = 3

In [ ]:
# Open file  
with open('../POLITICS_finetuning/processed_data.csv') as file_obj: 
    # Create reader object by passing the file  
    # object to reader method 
    reader_obj = csv.reader(file_obj) 
    with open('../POLITICS_finetuning/summariesAndBiases.csv', 'w') as f:  
        # Initialize writer object:
        writer_obj = csv.writer(f)
        # The fields of this file are
        # ['title', 'body', 'stance']
        # Iterate over each row in the .csv,
        # skipping the first row (pertaining to field / column)
        next(reader_obj)
        for row in reader_obj: 
            # Store title:
            title = row[0]
            # Store article:
            article = row[1]
            # Ensuring that articles fit within maximum length
            # article = article[:llm.tokenizer.model_max_length] 
            # For each stance:
            for stance in ['left', 'center', 'right']:
                for i in range(num_summaries):
                    summary = await gen_summary(llm_model_name, llm, bias_model, TwistModel, article, stance)
                    # For each summary, predict its bias:
                    pred_bias, logits = bias_model(summary)
                    writer_obj.writerows([title, summary, pred_bias, stance])
                    
    writer_obj.close()
    
reader.close()

